In [10]:
!pip install anthropic python-dotenv

In [11]:
from dotenv import load_dotenv
import json
load_dotenv()

True

In [ ]:
from anthropic import Anthropic
from statistics import mean

client = Anthropic()
model = "claude-haiku-4-5"

messages = []

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 500,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case['task']}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_prompt(test_case):
    prompt = f"""
    Solve the following task:

    {test_case['task']}
    """
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

def run_test_case(test_case):
    
    response = run_prompt(test_case)
    
    # grading

    evaluation = grade_by_model(test_case, response)
    score = evaluation.get("score", 0)
    reasoning = evaluation.get("reasoning", "")

    return {
        "output": response,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    print(f"Average score: {mean([r['score'] for r in results])}")
    return results

In [13]:
with open('008-dataset.json', 'r') as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))


JSONDecodeError: Unterminated string starting at: line 13 column 16 (char 613)